# BOLSTER's probe size and step size per epoch

BOLSTER drives SPSA on all weights with **one scalar Kalman filter** that is advanced once per epoch. Nothing in it is
tuned: the filter is initialised from the trained clean checkpoint $\theta_0$ and the epoch budget $E$ of the backprop
baseline (`SPSA_EPOCHS` in each sweep notebook):

$$P_0 = V\,\mathrm{RMS}(\theta_0),\qquad V = \mathrm{Var}_g\big(\mathrm{RMS}(\theta_{0,g})\big),\qquad R = E\,P_0,\qquad Q = 0,$$

where $g$ runs over the parameter-owning layers (conv / linear / batch-norm). Each epoch $e = 1 \dots E$:

$$P^-_e = P_{e-1} + Q,\qquad K_e = \frac{P^-_e}{P^-_e + R},\qquad c_e = P^-_e,\qquad a_e = K_e\,c_e,\qquad P_e = (1 - K_e)\,P^-_e,$$

and every SPSA step of that epoch uses the probe $c_e$ and the gain $K_e$:
$\theta \leftarrow \theta - K_e\,\frac{L(\theta + c_e d) - L(\theta - c_e d)}{2}\,d$, i.e. a step $a_e = K_e c_e$ along the
SPSA gradient estimate $\frac{L(\theta + c_e d) - L(\theta - c_e d)}{2 c_e}\,d$.
With $Q = 0$ the closed forms are $K_e = 1/(E+e)$, $c_e = P_0 E/(E+e-1)$ and $P_E = P_0/2$: the gain and the probe both
about halve over the run (so $a_e$ falls about four-fold), at a scale that the checkpoint itself sets.

This notebook plots the two quantities that set each step, **the gain $K_e$ and the probe $c_e = P^-_e$**, one figure per
benchmark. $K_e$ depends only on the epoch budget $E$; $c_e$ also carries the checkpoint's scale $P_0$. $P_0$ is recomputed
from each benchmark's clean checkpoint (bundled in `weights_baselines/`) with the sweep notebooks' rule, and the schedule
with the same recursion the runner uses; $E$ and $Q$, the settings of the reported runs, are written in the configuration
cell. The notebook reads no other file.

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

In [ ]:
FONT_FAMILY         = "DejaVu Sans"   # e.g. "serif", "Times New Roman", "Arial", ...
BASE_FONTSIZE       = 12              # default text size (titles, legend title, ...)
AXIS_LABEL_FONTSIZE = 19              # x/y axis label size
AXIS_TICK_FONTSIZE  = 16              # x/y tick label size
plt.rcParams.update({
    "font.family":     FONT_FAMILY,
    "font.size":       BASE_FONTSIZE,
    "axes.labelsize":  AXIS_LABEL_FONTSIZE,
    "axes.titlesize":  BASE_FONTSIZE + 1,
    "xtick.labelsize": AXIS_TICK_FONTSIZE,
    "ytick.labelsize": AXIS_TICK_FONTSIZE,
    "legend.fontsize": BASE_FONTSIZE - 3,
})

## Configuration

Presentation choices, the checkpoint location, and the two settings of the reported runs ($E$ and $Q$). $P_0$ is computed from each benchmark's clean checkpoint.

In [ ]:
# Model folder -> panel title. The order here is the subplot order.
MODELS = {
    "lenet5_mnist":             "LeNet5/MNIST",
    "resnet20_cifar10":         "ResNet20/CIFAR10",
    "mobilenetv1_tinyimagenet": "MobileNet-v1/Tiny ImageNet",
}

CLEAN_CHECKPOINT = "weights_baselines/full_retrain_var_0.0.pth"   # theta_0: the trained 0%-noise model (per model folder)

# The settings of the reported BOLSTER runs (SPSA_EPOCHS and KF_Q_REL in the sweep notebooks), keyed by model folder.
EPOCHS = {                # E: the epoch budget
    "lenet5_mnist":             10,
    "resnet20_cifar10":         200,
    "mobilenetv1_tinyimagenet": 300,
}
Q_REL = 0.0               # Q = Q_REL * P0: no process noise

STEP_COLOR  = "#D62728"   # gain K_e                          (left axis)
PROBE_COLOR = "#1F5FA8"   # probe c_e = P^-_e                 (right axis)

FIGSIZE     = (6.2, 4.2)
OUTPUT_DIR  = Path("figures")                      # one file per benchmark: bolster_kalman_schedule_<model folder>.png
OUTPUT_STEM = "bolster_kalman_schedule"

## The filter, as the sweep notebooks define it

`kf_init_from_state` reproduces `kf_init_from_checkpoint` of the sweeps: every module that owns parameters is one layer
(batch-norm running statistics are buffers, not parameters, and are excluded), $\mathrm{rms}_g$ is the RMS of that layer's
parameters, $V$ their population variance and $P_0 = V\,\mathrm{RMS}(\theta_0)$. `kf_plan` is the runner's `_kf_plan`.

In [ ]:
_BUFFER_SUFFIXES = ("running_mean", "running_var", "num_batches_tracked")


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Walk up from `start` until a folder is found that contains the model folders."""
    for candidate in (start, *start.parents):
        if all((candidate / m).is_dir() for m in MODELS):
            return candidate
    raise FileNotFoundError(f"model folders {list(MODELS)} not found in {start} or its parents")


def kf_init_from_state(state: dict) -> dict:
    """V = Var_g(rms_g) over the parameter-owning layers, RMS = RMS(theta), P0 = V * RMS  (float64, as in the sweeps)."""
    layers = {}
    for key, value in state.items():
        if not torch.is_tensor(value) or not value.dtype.is_floating_point or key.endswith(_BUFFER_SUFFIXES):
            continue                                            # buffers (BN statistics) are not parameters
        layers.setdefault(key.rsplit(".", 1)[0], []).append(value.detach().to(torch.float64).reshape(-1))
    flat_g = [torch.cat(v) for v in layers.values()]
    rms_g  = torch.stack([torch.sqrt((t * t).mean()) for t in flat_g])
    theta  = torch.cat(flat_g)
    rms_all = torch.sqrt((theta * theta).mean())
    assert bool(torch.isfinite(rms_g).all()) and bool((rms_g > 0).all()), "a layer with all-zero or non-finite weights"
    V = rms_g.var(unbiased=False)                                 # population variance over the G layers
    return dict(V=float(V), rms_all=float(rms_all), P0=float(V * rms_all), G=len(layers), n_params=int(theta.numel()),
                rms_g=rms_g, layer_names=list(layers))


def kf_plan(P0: float, Q: float, R: float, epochs: int) -> dict:
    """The filter over `epochs` epochs: prior P^-, gain K, probe c = P^-, step a = K c, posterior P (per epoch)."""
    P = float(P0)
    rows = []
    for _ in range(int(epochs)):
        Pm = P + Q                     # predict
        K  = Pm / (Pm + R)             # gain  (H = 1)
        rows.append((Pm, K, Pm, K * Pm))
        P  = (1.0 - K) * Pm            # update
    arr = np.asarray(rows, dtype=np.float64)
    return dict(epoch=np.arange(1, int(epochs) + 1), P_prior=arr[:, 0], K=arr[:, 1], c=arr[:, 2], a=arr[:, 3],
                P_post=np.append(arr[1:, 0], P) - Q, P_end=P)


## Read the checkpoints and run the schedule

In [ ]:
PROJECT_ROOT = find_project_root()
print(f"project root: {PROJECT_ROOT}\n")

runs = {}
for model_dir, title in MODELS.items():
    mdir = PROJECT_ROOT / model_dir
    ckpt = mdir / CLEAN_CHECKPOINT
    state = torch.load(ckpt, map_location="cpu", weights_only=False)
    kf0 = kf_init_from_state(state)
    E = EPOCHS[model_dir]
    q_rel = Q_REL
    P0, Q, R = kf0["P0"], q_rel * kf0["P0"], float(E) * kf0["P0"]
    plan = kf_plan(P0, Q, R, E)
    if Q == 0.0:   # closed forms hold with no process noise: K_e = 1/(E+e), c_e = P0 E/(E+e-1), P_E = P0/2
        e = plan["epoch"]
        assert np.allclose(plan["K"], 1.0 / (E + e)) and np.allclose(plan["c"], P0 * E / (E + e - 1))
        assert np.isclose(plan["P_end"], P0 / 2)
    runs[model_dir] = dict(title=title, kf0=kf0, E=E, Q=Q, R=R, plan=plan, ckpt=ckpt.relative_to(PROJECT_ROOT))
    print(f"== {title}   ({ckpt.relative_to(PROJECT_ROOT)})")
    print(f"   {kf0['n_params']:,} parameters in {kf0['G']} layers;  RMS(theta0) = {kf0['rms_all']:.4g},  "
          f"V = Var_g(rms_g) = {kf0['V']:.4g}  ->  P0 = V RMS = {P0:.4g}   (c_1/RMS = {P0 / kf0['rms_all']:.3%})")
    print(f"   E = {E} epochs,  R = E P0 = {R:.4g},  Q = {q_rel:g} P0 = {Q:.4g}")
    print(f"   epoch 1 -> {E}:  K {plan['K'][0]:.3e} -> {plan['K'][-1]:.3e};  c = P^- {plan['c'][0]:.3e} -> {plan['c'][-1]:.3e}  "
          f"(x{plan['c'][0] / plan['c'][-1]:.2f});  a = K c {plan['a'][0]:.3e} -> {plan['a'][-1]:.3e}  (x{plan['a'][0] / plan['a'][-1]:.2f});  "
          f"P ends at {plan['P_end'] / P0:.3f} P0\n")

## Plot: gain $K_e$ and probe $c_e$ per epoch, one figure per benchmark

In [ ]:
def _sci(ax_axis):
    fmt = ScalarFormatter(useMathText=True)
    fmt.set_powerlimits((-2, 2))
    ax_axis.set_major_formatter(fmt)


# K_e is close to c_e / R, so with each axis scaled to its own maximum the two curves would lie on top of each other.
# Each axis is scaled instead to put its curve in its own band (both axes still start at 0): the height, as a
# fraction of the axes, of each curve's first-epoch value.
STEP_PEAK  = 0.85    # K_e: upper band
PROBE_PEAK = 0.45    # c_e: lower band
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for model_dir, r in runs.items():
    plan, E, P0, kf0 = r["plan"], r["E"], r["kf0"]["P0"], r["kf0"]
    e = plan["epoch"]
    marker = None

    fig, ax = plt.subplots(figsize=FIGSIZE)

    # ---- left axis: the gain K_e ----
    ax.plot(e, plan["K"], color=STEP_COLOR, lw=2.3, marker=marker, ms=5, zorder=4, label="Step gain, $K_e$")
    ax.set_ylim(0, plan["K"].max() / STEP_PEAK)
    ax.set_xlim(1, E)
    ax.set_xlabel("Epochs [#]")
    ax.set_ylabel("Step gain, $K_e$")
    ax.tick_params(axis="y", colors=STEP_COLOR)
    _sci(ax.yaxis)
    ax.yaxis.get_offset_text().set_color(STEP_COLOR)
    ax.grid(True, alpha=0.22)

    # ---- right axis: the probe c_e = P^-_e (the prior covariance) ----
    ax2 = ax.twinx()
    ax2.plot(e, plan["c"], color=PROBE_COLOR, lw=2.3, ls="--", marker=marker, ms=5, zorder=3, label="Probe gain, $P_{e|e-1}$")
    ax2.set_ylim(0, plan["c"].max() / PROBE_PEAK)
    # two lines: at the larger label font, one line would be taller than the axis
    ax2.set_ylabel("Probe gain, $P_{e|e-1}$", color=PROBE_COLOR)
    ax2.tick_params(axis="y", colors=PROBE_COLOR)
    _sci(ax2.yaxis)
    ax2.yaxis.get_offset_text().set_color(PROBE_COLOR)

    # ---- the benchmark name, boxed in the upper-right corner (a text box pads evenly; a title-only legend does not) ----
    # ax.text(0.975, 0.965, r["title"], transform=ax.transAxes, fontsize=11, ha="right", va="top", zorder=6,
    #         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="0.6", alpha=0.95))
    
    # ---- combined legend with benchmark name as title ----
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    legend = ax.legend(
        lines1 + lines2,
        labels1 + labels2,
        title=r["title"],
        loc="upper right",
        fontsize=12,
        title_fontsize=12,
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        edgecolor="0.6",
        borderpad=0.4,
        labelspacing=0.5
    )

    legend.get_title()

    fig.tight_layout()
    out = OUTPUT_DIR / f"{OUTPUT_STEM}_{model_dir}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print(f"saved {out.resolve()}")
    plt.show()